<a href="https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page, on one specific date, for one client. (A "page-day.") I'm using a mid-panel month, March 2026 (month=2026-03) from fact_content_daily_performance — not the first month (data may still be ramping up) and not the last, June 2026 (sealed for testing later).

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import pandas as pd

login(token=userdata.get('HF_TOKEN'))

path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/"
march = pd.read_parquet(path)

print(f"Rows loaded: {len(march):,}")
print(f"Date range: {march['report_date'].min()} to {march['report_date'].max()}")

dupes = march.duplicated(subset=["content_hash_id", "report_date"]).sum()
print(f"Duplicate page-day rows: {dupes}")

Rows loaded: 9,841,378
Date range: 2026-03-01 to 2026-03-31
Duplicate page-day rows: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (knowable before the decision moment — describes a page's past behavior):
gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_engaged_sessions, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, sessions_organic, sessions_ai. Each describes something already true about the page before the decision point.

Label (the answer I'm predicting): not a separate column — built by comparing gsc_impressions across two time windows: an earlier "past" window used as a feature, and a later "future" window used to compute the label (did it drop 20%+?). Same raw column, two jobs depending on window — this is the leakage risk I'll test in Task 4.

Context (identifies the row, not a signal fed to the model):
report_date, client_hash_id, content_hash_id.

Excluded (deliberately skipped this round, with why):

ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — seven sparse AI-referral columns; real signal eventually, too much complexity before the core model is validated.
scroll_events, ga4_total_engagement_sec, sessions_direct, sessions_referral, sessions_social, sessions_paid — engagement-depth detail; my question is about search visibility, not on-page engagement.
gsc_sum_position — redundant with gsc_avg_position, the cleaner averaged version of the same data.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Availability check: GSC data
gsc_available = march[march["gsc_data_available"] == True]
print(f"Rows with gsc_data_available == True: {len(gsc_available):,} out of {len(march):,} ({len(gsc_available)/len(march):.1%})")

# Availability check: GA4 data
ga4_available = march[march["ga4_data_available"] == True]
print(f"Rows with ga4_data_available == True: {len(ga4_available):,} out of {len(march):,} ({len(ga4_available)/len(march):.1%})")

# Show what the "missing" values actually look like — proving NULL isn't the same as False
print(f"\nga4_data_available value counts (including NULLs):")
print(march["ga4_data_available"].value_counts(dropna=False))

Rows with gsc_data_available == True: 3,611,061 out of 9,841,378 (36.7%)
Rows with ga4_data_available == True: 413,966 out of 9,841,378 (4.2%)

ga4_data_available value counts (including NULLs):
ga4_data_available
False    6408671
None     3018741
True      413966
Name: count, dtype: int64


Availability check on March 2026: gsc_data_available is True for 36.7% of rows; ga4_data_available is True for only 4.2%. The breakdown of ga4_data_available shows three distinct categories — False (65.1%, confirmed no GA4), None (30.7%, unknown/unmeasured), True (4.2%, trustworthy). Filtering with == False alone would have missed the None rows, wrongly treating "unknown" the same as "excluded" — that's exactly the IS TRUE trap the card warned about. Given how sparse GA4 is, my features should lean on GSC signals primarily, with GA4 treated as optional/supplementary.

In [ ]:
features = march.groupby("content_hash_id").agg(
    total_impressions=("gsc_impressions", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    total_pageviews=("ga4_pageviews", "sum"),
    pct_days_gsc_available=("gsc_data_available", "mean"),
    total_organic_sessions=("sessions_organic", "sum"),
).reset_index()

print(f"Feature frame: {len(features):,} pages, {features.shape[1]-1} features")
display(features.head())

Feature frame: 331,437 pages, 5 features


,content_hash_id,total_impressions,avg_position,total_pageviews,pct_days_gsc_available,total_organic_sessions
0,content_000005d4ced12088,86,72.854861,0.0,0.774194,0.0
1,content_00001e488b74b799,0,NaN,0.0,0.000000,0.0
2,content_00007bd2985b77c3,47,5.269565,0.0,0.741935,0.0
3,content_00008950670cb6b5,0,NaN,2.0,0.000000,0.0
4,content_0000a348850eb1fc,0,NaN,1.0,0.000000,0.0


Five features (knowable at the decision moment):

total_impressions — knowable because it sums the page's GSC impressions already observed during March, before any future outcome.
avg_position — knowable because it's the page's average Google ranking already measured in March.
total_pageviews — knowable because it's GA4 pageviews already recorded (where available).
pct_days_gsc_available — knowable because it reflects the page/client's connection status during March, not future performance.
total_organic_sessions — knowable because it's organic traffic already observed in March.

In [ ]:
import datetime
cutoff = datetime.date(2026, 3, 15)

half1 = march[march["report_date"] <= cutoff].groupby("content_hash_id")["gsc_impressions"].sum()
half2 = march[march["report_date"] > cutoff].groupby("content_hash_id")["gsc_impressions"].sum()

print(f"Pages in first half: {len(half1):,}")
print(f"Pages in second half: {len(half2):,}")

Pages in first half: 319,759
Pages in second half: 331,436


In [ ]:
label_df = pd.DataFrame({"impressions_h1": half1, "impressions_h2": half2}).dropna()
label_df = label_df[label_df["impressions_h1"] >= 50]  # enough volume to measure a real drop

label_df["faded"] = ((label_df["impressions_h2"] - label_df["impressions_h1"]) / label_df["impressions_h1"] <= -0.20).astype(int)

print(f"Pages with enough volume in both halves: {len(label_df):,}")
print(f"Share labeled 'faded': {label_df['faded'].mean():.1%}")
label_df.head()

Pages with enough volume in both halves: 92,548
Share labeled 'faded': 28.7%


,impressions_h1,impressions_h2,faded
content_hash_id,,,
content_00014efc121d911d,53.0,63.0,0
content_000184dde41afe75,2405.0,2480.0,0
content_0002bd310bf01f15,66.0,121.0,0
content_00032be2df0005ca,301.0,601.0,0
content_00033c286cc93446,185.0,224.0,0


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
h1_data = march[march["report_date"] <= datetime.date(2026, 3, 15)]

honest_features = h1_data.groupby("content_hash_id").agg(
    total_impressions=("gsc_impressions", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    total_pageviews=("ga4_pageviews", "sum"),
    pct_days_gsc_available=("gsc_data_available", "mean"),
    total_organic_sessions=("sessions_organic", "sum"),
)

X_honest2 = honest_features.reindex(label_df.index).fillna(0)
y = label_df["faded"]

honest_score2 = cross_val_score(DecisionTreeClassifier(max_depth=3), X_honest2, y, cv=3).mean()
print(f"TRUE honest score (features from half1 only): {honest_score2:.3f}")

TRUE honest score (features from half1 only): 0.713


In [ ]:
data2 = X_honest2.copy()
data2["leak_impressions_h2"] = label_df["impressions_h2"]

leaked_score2 = cross_val_score(DecisionTreeClassifier(max_depth=3), data2, y, cv=3).mean()
print(f"Leaked score (with half2 column added): {leaked_score2:.3f}")
print(f"Honest score (no leak):                 {honest_score2:.3f}")
print(f"Jump: +{leaked_score2 - honest_score2:.3f}")

Leaked score (with half2 column added): 0.776
Honest score (no leak):                 0.713
Jump: +0.063


The trap: I built a rough label by comparing each page's impressions in the first half of March (half1) vs. the second half (half2) — did it drop 20% or more? Then I trained a small decision tree on five honest features, all computed from half1 only (the true "past"). Honest score: 0.713.

Next, I deliberately added one leaked column: leak_impressions_h2, which is literally impressions_h2 — the same raw number my label was calculated from. Retraining with this column included pushed the score to 0.776, a jump of +0.063.

The jump is real but more modest than dramatic, and I think I understand why: my label is a ratio (percent change between h1 and h2), and a shallow decision tree (max_depth=3) can only ask one-column-at-a-time yes/no questions — it can't easily combine h1 and the leaked h2 into that exact ratio rule. So even with the answer's raw ingredient sitting right there, a simple model can't fully exploit it. A more complex model (deeper tree, or one built to combine features) would likely leak much more severely — which is itself a useful lesson: leakage risk isn't fixed, it depends on how much a model is capable of exploiting the leaked signal.

Either way, this confirms the mechanism: impressions_h2 is the same window my label was computed from, so including it as a feature let the model partially "peek" at the future — I removed it and kept the honest 0.713 as my real, trustworthy score.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell me the true GA4 picture for most pages: only 4.2% of March rows have ga4_data_available == True, with 30.7% unknown (None) and 65.1% confirmed unavailable. This means GA4-based features (sessions, engagement) would only apply to a small, possibly unrepresentative subset of pages — likely clients who connected GA4 early or fully. GSC coverage is better (36.7%) but still leaves most rows unusable for GSC-based signals too.

This also connects to the warehouse's known "unbalanced panel" issue: different clients have different gsc_data_start/ga4_data_start dates, so a client's early rows may show up as unavailable simply because their connection hadn't started yet — not because the page had no real activity. Any comparison across clients has to account for this, or risk mistaking "data not yet connected" for "page not performing."

Finally, this is a single mid-panel month — one 31-day window. It can't tell me about seasonality, longer trends, or how a page's fade unfolds over multiple months; that requires the full daily history across many months, which is beyond this notebook's scope.

In [ ]:
# Confirming the limitation above with the same query output from Section 3
print(f"GA4 availability breakdown (March 2026):")
print(march["ga4_data_available"].value_counts(dropna=False, normalize=True).mul(100).round(1))

GA4 availability breakdown (March 2026):
ga4_data_available
False    65.1
None     30.7
True      4.2
Name: proportion, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.